# O3A - Skill-Weighted Annual Projection Ensemble

This notebook implements the first defensible stage of Objective 3.

It uses O2-passing CMIP6 models and annual CMIP6 projection files to create zone-bias-corrected, skill-weighted ensemble projections for precipitation, Tmax, and Tmin under SSP2-4.5 and SSP5-8.5.

Scope note: this is O3A, an annual projection ensemble stage. It does not claim completion of the full PINN/RF/XGBoost/Ridge/BMA system. Those should be handled as later O3B/O3C/O7 steps after this baseline ensemble is checked.

Main outputs are saved to `output/o3a_skill_weighted_ensemble/`.

In [ ]:
# Cell 1 - Imports and configuration
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'output').exists() and (PROJECT_ROOT.parent / 'output').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUT_ROOT = PROJECT_ROOT / 'output' / 'o3a_skill_weighted_ensemble'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
NC_DIR = OUT_ROOT / 'netcdf'
TIF_DIR = OUT_ROOT / 'geotiff'
LOG_DIR = OUT_ROOT / 'logs'
SUMMARY_PATH = OUT_ROOT / 'O3A_SKILL_WEIGHTED_ENSEMBLE_SUMMARY.md'
for d in [TABLE_DIR, FIG_DIR, NC_DIR, TIF_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CMIP6_ROOT = PROJECT_ROOT / 'output' / 'cmip6'
O2_TABLE_DIR = PROJECT_ROOT / 'output' / 'cmip6_eval' / 'tables'
ZONE_NC = PROJECT_ROOT / 'output' / 'zones' / 'hydroclimatic_zones_SA.nc'
VARIABLES = ['pr', 'tasmax', 'tasmin']
SCENARIOS = ['ssp245', 'ssp585']
HISTORICAL = 'historical'
BASELINE_YEARS = (1985, 2014)
FUTURE_PERIODS = {'near_future': (2021, 2040), 'mid_future': (2041, 2070), 'far_future': (2071, 2100)}
SKIP_EXISTING = True
SAVE_CORRECTED_MODEL_FILES = False
print(f'[INFO] Project root: {PROJECT_ROOT}')
print(f'[INFO] Output root: {OUT_ROOT}')

In [ ]:
# Cell 2 - Helper functions
def find_coord_name(obj, candidates):
    for c in candidates:
        if c in obj.coords or c in obj.dims:
            return c
    lowered = {str(k).lower(): k for k in list(obj.coords) + list(obj.dims)}
    for c in candidates:
        if c.lower() in lowered:
            return lowered[c.lower()]
    return None

def pick_data_var(ds, preferred):
    if preferred in ds.data_vars:
        return preferred
    for name in ds.data_vars:
        if preferred.lower() in name.lower():
            return name
    data_vars = list(ds.data_vars)
    if not data_vars:
        raise ValueError('No data variables found in dataset')
    return data_vars[0]

def standardise_lat_lon(da):
    lat = find_coord_name(da, ['lat', 'latitude', 'y'])
    lon = find_coord_name(da, ['lon', 'longitude', 'x'])
    ren = {}
    if lat and lat != 'lat': ren[lat] = 'lat'
    if lon and lon != 'lon': ren[lon] = 'lon'
    if ren: da = da.rename(ren)
    if 'lat' not in da.dims or 'lon' not in da.dims:
        raise ValueError(f'Could not identify lat/lon dims in {da.dims}')
    return da

def load_cmip6_annual(model, scenario, variable):
    suffix = '1985_2014' if scenario == HISTORICAL else '2015_2100'
    path = CMIP6_ROOT / scenario / model / f'{model}_{variable}_{scenario}_{suffix}.nc'
    if not path.exists():
        raise FileNotFoundError(path)
    ds = xr.open_dataset(path)
    da = standardise_lat_lon(ds[pick_data_var(ds, variable)])
    if 'year' not in da.dims:
        time_name = find_coord_name(da, ['time'])
        if time_name is None:
            raise ValueError(f'No time/year dimension found in {path}')
        if time_name != 'time':
            da = da.rename({time_name: 'time'})
        if np.issubdtype(da['time'].dtype, np.datetime64):
            n_steps = da.sizes['time']
            if variable == 'pr' and n_steps > 100:
                da = da.groupby('time.year').sum('time', skipna=True)
            elif n_steps > 100:
                da = da.groupby('time.year').mean('time', skipna=True)
            else:
                da = da.assign_coords(year=da['time'].dt.year).swap_dims({'time':'year'}).drop_vars('time')
        else:
            da = da.rename({'time':'year'})
    da = da.sortby('year')
    da.name = variable
    return da

def load_zone_mask_like(target_da):
    zds = xr.open_dataset(ZONE_NC)
    z = standardise_lat_lon(zds[pick_data_var(zds, 'zone')])
    z_like = z.interp(lat=target_da['lat'], lon=target_da['lon'], method='nearest').round().astype('int16')
    z_like.name = 'zone'
    return z_like

def get_model_file_exists(model, scenario, variable):
    suffix = '1985_2014' if scenario == HISTORICAL else '2015_2100'
    return (CMIP6_ROOT / scenario / model / f'{model}_{variable}_{scenario}_{suffix}.nc').exists()

def weighted_quantile(values, weights, quantile):
    values = np.asarray(values, dtype=float); weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if mask.sum() == 0: return np.nan
    values = values[mask]; weights = weights[mask]
    sorter = np.argsort(values); values = values[sorter]; weights = weights[sorter]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(quantile, cdf, values)

def xr_weighted_quantile(stack_da, weights_da, quantile):
    return xr.apply_ufunc(weighted_quantile, stack_da, weights_da, input_core_dims=[['model'], ['model']], output_core_dims=[[]], kwargs={'quantile': quantile}, vectorize=True, dask='parallelized', output_dtypes=[float])

def safe_to_geotiff(da, path):
    try:
        import rioxarray  # noqa
        out = da.copy()
        if 'lat' in out.dims and 'lon' in out.dims:
            out = out.rio.set_spatial_dims(x_dim='lon', y_dim='lat', inplace=False)
        out = out.rio.write_crs('EPSG:4326', inplace=False)
        out.rio.to_raster(path)
        return True, ''
    except Exception as e:
        return False, str(e)
print('[OK] helper functions loaded')

In [ ]:
# Cell 3 - Build O2-passing model pools and weights
ranking = pd.read_csv(O2_TABLE_DIR / 'model_ranking.csv')
suitability = pd.read_csv(O2_TABLE_DIR / 'model_suitability_matrix.csv')
var_skill = pd.read_csv(O2_TABLE_DIR / 'variable_specific_model_skill_scores.csv')
passed_models = suitability.loc[suitability['in_ensemble_pool'].astype(str).str.lower() == 'true', 'model'].tolist()
print('[INFO] O2-passing models:', passed_models)
rows = []
for variable in VARIABLES:
    for model in passed_models:
        if not all(get_model_file_exists(model, sc, variable) for sc in [HISTORICAL] + SCENARIOS):
            continue
        score_row = var_skill[(var_skill['model'] == model) & (var_skill['variable'] == variable)]
        score = float(score_row['variable_skill_score'].iloc[0]) if len(score_row) else float(ranking.loc[ranking['model'] == model, 'composite_score'].iloc[0])
        rows.append({'variable': variable, 'model': model, 'raw_skill_score': score})
pool = pd.DataFrame(rows)
pool['positive_skill_score'] = pool['raw_skill_score'].clip(lower=0.001)
pool['weight'] = pool.groupby('variable')['positive_skill_score'].transform(lambda x: x / x.sum())
pool = pool.sort_values(['variable', 'weight'], ascending=[True, False]).reset_index(drop=True)
pool.to_csv(TABLE_DIR / 'o3a_model_pools_and_weights.csv', index=False)
print(pool)
print(f'[OK] saved {TABLE_DIR / "o3a_model_pools_and_weights.csv"}')

In [ ]:
# Cell 4 - Build annual zone-bias-correction factors
# The O2 time-series tables may store years as either `year` or datetime-like `time`.
# This cell standardises them before calculating zone-specific correction factors.

def ensure_year_column(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    if 'year' not in df.columns:
        if 'time' not in df.columns:
            raise KeyError(f'Expected either year or time column. Columns found: {list(df.columns)}')
        df['year'] = pd.to_datetime(df['time']).dt.year
    df['year'] = df['year'].astype(int)
    return df

era5_zone = ensure_year_column(pd.read_csv(PROJECT_ROOT / 'output' / 'cmip6_eval' / 'timeseries' / 'era5_zone_annual.csv'))
cmip_zone = ensure_year_column(pd.read_csv(PROJECT_ROOT / 'output' / 'cmip6_eval' / 'timeseries' / 'cmip6_historical_zone_annual.csv'))

era5_base = era5_zone[(era5_zone['year'] >= BASELINE_YEARS[0]) & (era5_zone['year'] <= BASELINE_YEARS[1])]
cmip_base = cmip_zone[(cmip_zone['year'] >= BASELINE_YEARS[0]) & (cmip_zone['year'] <= BASELINE_YEARS[1])]

rows = []
for _, p in pool.iterrows():
    model = p['model']
    variable = p['variable']
    obs = era5_base[era5_base['variable'] == variable].groupby('zone', as_index=False)['value'].mean().rename(columns={'value':'era5_mean'})
    sim = cmip_base[(cmip_base['model'] == model) & (cmip_base['variable'] == variable)].groupby('zone', as_index=False)['value'].mean().rename(columns={'value':'historical_model_mean'})
    merged = obs.merge(sim, on='zone', how='inner')
    for _, r in merged.iterrows():
        if variable == 'pr':
            factor = r['era5_mean'] / r['historical_model_mean'] if r['historical_model_mean'] != 0 else np.nan
            offset = np.nan
            method = 'multiplicative_ratio'
        else:
            factor = np.nan
            offset = r['era5_mean'] - r['historical_model_mean']
            method = 'additive_offset_degC'
        rows.append({
            'model': model,
            'variable': variable,
            'zone': r['zone'],
            'era5_mean': r['era5_mean'],
            'historical_model_mean': r['historical_model_mean'],
            'factor': factor,
            'offset': offset,
            'method': method,
        })

correction = pd.DataFrame(rows)
if correction.empty:
    raise RuntimeError('No bias-correction factors were created. Check model pool and O2 time-series tables.')
correction.to_csv(TABLE_DIR / 'o3a_zone_bias_correction_factors.csv', index=False)
print(correction.head(20))
print(f'[OK] saved {TABLE_DIR / "o3a_zone_bias_correction_factors.csv"}')

In [ ]:
# Cell 5 - Correction and ensemble functions
def apply_zone_bias_correction(da, model, variable, zone_mask):
    sub = correction[(correction['model'] == model) & (correction['variable'] == variable)]
    if sub.empty:
        raise ValueError(f'No correction factors for {model} {variable}')
    corrected = xr.full_like(da, np.nan, dtype='float32')
    for _, row in sub.iterrows():
        zone_label = str(row['zone'])
        zone_num = int(zone_label.replace('Z','')) if zone_label.startswith('Z') else int(row['zone'])
        mask = zone_mask == zone_num
        val = (da * float(row['factor'])).clip(min=0) if variable == 'pr' else da + float(row['offset'])
        corrected = xr.where(mask, val, corrected)
    corrected.name = variable
    corrected.attrs.update(da.attrs)
    corrected.attrs['o3a_bias_correction'] = 'zone-specific annual correction using ERA5 and CMIP6 historical 1985-2014 means'
    return corrected

def make_ensemble(variable, scenario):
    out_path = NC_DIR / scenario / f'o3a_{variable}_{scenario}_skill_weighted_ensemble_annual.nc'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if SKIP_EXISTING and out_path.exists():
        print(f'[SKIP] {out_path}')
        return xr.open_dataset(out_path)
    corrected_members = []
    used_rows = []
    for _, row in pool[pool['variable'] == variable].iterrows():
        model = row['model']
        try:
            da = load_cmip6_annual(model, scenario, variable)
            zone_mask = load_zone_mask_like(da)
            cda = apply_zone_bias_correction(da, model, variable, zone_mask).assign_coords(model=model).expand_dims('model')
            corrected_members.append(cda)
            used_rows.append({'scenario':scenario,'variable':variable,'model':model,'weight':row['weight']})
            if SAVE_CORRECTED_MODEL_FILES:
                model_dir = NC_DIR / 'corrected_members' / scenario / model
                model_dir.mkdir(parents=True, exist_ok=True)
                cda.to_dataset(name=variable).to_netcdf(model_dir / f'{model}_{variable}_{scenario}_annual_zone_corrected.nc')
            print(f'[OK] loaded/corrected {model} | {scenario} | {variable}')
        except Exception as e:
            with open(LOG_DIR / 'o3a_failed_members.txt', 'a', encoding='utf-8') as f:
                f.write(f'{scenario}|{variable}|{model}|{type(e).__name__}: {e}\n')
            print(f'[FAIL] {scenario}|{variable}|{model}|{type(e).__name__}: {e}')
    if not corrected_members:
        raise RuntimeError(f'No valid members for {scenario} {variable}')
    stack = xr.concat(corrected_members, dim='model')
    used = pd.DataFrame(used_rows)
    used['weight'] = used['weight'] / used['weight'].sum()
    weights_da = xr.DataArray(used.set_index('model').loc[stack['model'].values, 'weight'].values, dims=['model'], coords={'model': stack['model'].values})
    ds = xr.Dataset({
        f'{variable}_mean': stack.weighted(weights_da).mean('model', skipna=True).astype('float32'),
        f'{variable}_p05': xr_weighted_quantile(stack, weights_da, 0.05).astype('float32'),
        f'{variable}_p95': xr_weighted_quantile(stack, weights_da, 0.95).astype('float32'),
        f'{variable}_n_models': stack.notnull().sum('model').astype('int16'),
    })
    ds.attrs['objective'] = 'O3A skill-weighted annual projection ensemble'
    ds.attrs['scenario'] = scenario
    ds.attrs['variable'] = variable
    ds.attrs['model_weighting'] = 'O2 variable-specific skill scores, normalized over available O2-passing models'
    ds.attrs['bias_correction'] = 'Zone-specific annual correction using ERA5 and CMIP6 historical 1985-2014 zone means'
    ds.to_netcdf(out_path)
    used.to_csv(TABLE_DIR / f'o3a_used_members_{scenario}_{variable}.csv', index=False)
    print(f'[OK] saved {out_path}')
    return ds
print('[OK] ensemble functions ready')

In [ ]:
# Cell 6 - Run O3A ensembles for historical baseline and SSP futures
ensemble_outputs = {}
for scenario in [HISTORICAL] + SCENARIOS:
    for variable in VARIABLES:
        print(f'\n=== {scenario} | {variable} ===')
        try:
            ensemble_outputs[(scenario, variable)] = make_ensemble(variable, scenario)
        except Exception as e:
            with open(LOG_DIR / 'o3a_failed_ensembles.txt', 'a', encoding='utf-8') as f:
                f.write(f'{scenario}|{variable}|{type(e).__name__}: {e}\n')
            print(f'[FAIL] ensemble {scenario}|{variable}|{type(e).__name__}: {e}')
print('[DONE] O3A ensemble processing cell finished')

In [ ]:
# Cell 7 - Zone annual means from ensemble outputs
zone_rows = []
for scenario in [HISTORICAL] + SCENARIOS:
    for variable in VARIABLES:
        path = NC_DIR / scenario / f'o3a_{variable}_{scenario}_skill_weighted_ensemble_annual.nc'
        if not path.exists(): continue
        ds = xr.open_dataset(path)
        zmask = load_zone_mask_like(ds[f'{variable}_mean'])
        zone_values = np.unique(zmask.values[np.isfinite(zmask.values)])
        for zone_num in sorted([int(z) for z in zone_values if z > 0]):
            mask = zmask == zone_num
            for stat_name, da_name in [('mean', f'{variable}_mean'), ('p05', f'{variable}_p05'), ('p95', f'{variable}_p95')]:
                zmean = ds[da_name].where(mask).mean(dim=('lat','lon'), skipna=True)
                for y, val in zip(zmean['year'].values, zmean.values):
                    zone_rows.append({'scenario':scenario,'variable':variable,'zone':f'Z{zone_num}','year':int(y),'stat':stat_name,'value':float(val) if np.isfinite(val) else np.nan})
zone_annual = pd.DataFrame(zone_rows)
zone_annual.to_csv(TABLE_DIR / 'o3a_zone_annual_ensemble_timeseries.csv', index=False)
print(zone_annual.head())
print(f'[OK] saved {TABLE_DIR / "o3a_zone_annual_ensemble_timeseries.csv"}')

In [ ]:
# Cell 8 - Period changes by scenario, variable, and zone
zone_annual = pd.read_csv(TABLE_DIR / 'o3a_zone_annual_ensemble_timeseries.csv')
rows = []
base = zone_annual[(zone_annual['scenario'] == HISTORICAL) & (zone_annual['stat'] == 'mean')]
base = base[(base['year'] >= BASELINE_YEARS[0]) & (base['year'] <= BASELINE_YEARS[1])]
base_mean = base.groupby(['variable','zone'], as_index=False)['value'].mean().rename(columns={'value':'baseline_1985_2014'})
for scenario in SCENARIOS:
    for period_name, (start, end) in FUTURE_PERIODS.items():
        fut = zone_annual[(zone_annual['scenario'] == scenario) & (zone_annual['stat'] == 'mean')]
        fut = fut[(fut['year'] >= start) & (fut['year'] <= end)]
        fut_mean = fut.groupby(['variable','zone'], as_index=False)['value'].mean().rename(columns={'value':'future_mean'})
        merged = base_mean.merge(fut_mean, on=['variable','zone'], how='inner')
        for _, r in merged.iterrows():
            absolute_change = r['future_mean'] - r['baseline_1985_2014']
            percent_change = absolute_change / r['baseline_1985_2014'] * 100 if r['variable'] == 'pr' and r['baseline_1985_2014'] != 0 else np.nan
            rows.append({'scenario':scenario,'period':period_name,'start_year':start,'end_year':end,'variable':r['variable'],'zone':r['zone'],'baseline_1985_2014':r['baseline_1985_2014'],'future_mean':r['future_mean'],'absolute_change':absolute_change,'percent_change_pr_only':percent_change})
changes = pd.DataFrame(rows)
changes.to_csv(TABLE_DIR / 'o3a_period_changes_by_zone.csv', index=False)
print(changes.head(20))
print(f'[OK] saved {TABLE_DIR / "o3a_period_changes_by_zone.csv"}')

In [ ]:
# Cell 9 - Spatial period-change maps and optional GeoTIFF export
spatial_rows = []
for scenario in SCENARIOS:
    for variable in VARIABLES:
        hist_path = NC_DIR / HISTORICAL / f'o3a_{variable}_{HISTORICAL}_skill_weighted_ensemble_annual.nc'
        fut_path = NC_DIR / scenario / f'o3a_{variable}_{scenario}_skill_weighted_ensemble_annual.nc'
        if not hist_path.exists() or not fut_path.exists(): continue
        hist = xr.open_dataset(hist_path)[f'{variable}_mean']
        fut = xr.open_dataset(fut_path)[f'{variable}_mean']
        baseline = hist.sel(year=slice(BASELINE_YEARS[0], BASELINE_YEARS[1])).mean('year', skipna=True)
        for period_name, (start, end) in FUTURE_PERIODS.items():
            out_nc = NC_DIR / 'period_changes' / scenario / variable / f'o3a_{variable}_{scenario}_{period_name}_change_vs_1985_2014.nc'
            out_nc.parent.mkdir(parents=True, exist_ok=True)
            if SKIP_EXISTING and out_nc.exists(): print(f'[SKIP] {out_nc}')
            else:
                future_mean = fut.sel(year=slice(start, end)).mean('year', skipna=True)
                abs_change = future_mean - baseline
                ds_out = xr.Dataset({'absolute_change': abs_change.astype('float32')})
                if variable == 'pr': ds_out['percent_change'] = ((abs_change / baseline) * 100).astype('float32')
                ds_out.attrs['baseline'] = '1985-2014 historical skill-weighted bias-corrected ensemble mean'
                ds_out.attrs['future_period'] = f'{start}-{end}'
                ds_out.to_netcdf(out_nc); print(f'[OK] {out_nc}')
            ds_saved = xr.open_dataset(out_nc)
            tif_path = TIF_DIR / scenario / variable / f'o3a_{variable}_{scenario}_{period_name}_absolute_change.tif'
            tif_path.parent.mkdir(parents=True, exist_ok=True)
            ok, err = safe_to_geotiff(ds_saved['absolute_change'], tif_path)
            spatial_rows.append({'scenario':scenario,'variable':variable,'period':period_name,'netcdf':str(out_nc),'geotiff_absolute_change':str(tif_path) if ok else '','geotiff_status':'ok' if ok else f'failed: {err}'})
spatial_inventory = pd.DataFrame(spatial_rows)
spatial_inventory.to_csv(TABLE_DIR / 'o3a_spatial_change_inventory.csv', index=False)
print(spatial_inventory.head())
print(f'[OK] saved {TABLE_DIR / "o3a_spatial_change_inventory.csv"}')

In [ ]:
# Cell 10 - Figures for manuscript screening
pool = pd.read_csv(TABLE_DIR / 'o3a_model_pools_and_weights.csv')
changes = pd.read_csv(TABLE_DIR / 'o3a_period_changes_by_zone.csv')
zone_annual = pd.read_csv(TABLE_DIR / 'o3a_zone_annual_ensemble_timeseries.csv')
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for ax, variable in zip(axes, VARIABLES):
    sub = pool[pool['variable'] == variable].sort_values('weight', ascending=True)
    ax.barh(sub['model'], sub['weight'], color='#4c78a8', edgecolor='black', linewidth=0.5)
    ax.set_title(variable.upper(), fontweight='bold'); ax.set_xlabel('Normalized weight', fontweight='bold'); ax.grid(axis='x', linestyle=':', alpha=0.5)
fig.suptitle('O3A Skill-Based Model Weights by Variable', fontsize=14, fontweight='bold')
fig.savefig(FIG_DIR / 'o3a_model_weights_by_variable.png', dpi=350, bbox_inches='tight'); fig.savefig(FIG_DIR / 'o3a_model_weights_by_variable.pdf', bbox_inches='tight'); plt.show()
for variable in VARIABLES:
    sub = changes[changes['variable'] == variable].copy()
    if sub.empty: continue
    value_col = 'percent_change_pr_only' if variable == 'pr' else 'absolute_change'
    ylabel = 'Precipitation change (%)' if variable == 'pr' else 'Temperature change (degC)'
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True, constrained_layout=True)
    for ax, scenario in zip(axes, SCENARIOS):
        ss = sub[sub['scenario'] == scenario]
        pivot = ss.pivot_table(index='zone', columns='period', values=value_col)
        pivot = pivot[[p for p in FUTURE_PERIODS.keys() if p in pivot.columns]]
        pivot.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.4)
        ax.axhline(0, color='black', linewidth=0.8); ax.set_title(scenario.upper(), fontweight='bold'); ax.set_xlabel('Hydroclimatic zone', fontweight='bold'); ax.set_ylabel(ylabel, fontweight='bold'); ax.grid(axis='y', linestyle=':', alpha=0.5); ax.legend(title='Period', fontsize=8)
    fig.suptitle(f'O3A Zone-Level Projected Change: {variable.upper()}', fontsize=14, fontweight='bold')
    fig.savefig(FIG_DIR / f'o3a_zone_period_changes_{variable}.png', dpi=350, bbox_inches='tight'); fig.savefig(FIG_DIR / f'o3a_zone_period_changes_{variable}.pdf', bbox_inches='tight'); plt.show()
for variable in VARIABLES:
    sub = zone_annual[(zone_annual['variable'] == variable) & (zone_annual['stat'] == 'mean')]
    if sub.empty: continue
    fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    for scenario in [HISTORICAL] + SCENARIOS:
        ss = sub[sub['scenario'] == scenario].groupby('year', as_index=False)['value'].mean()
        if len(ss): ax.plot(ss['year'], ss['value'], label=scenario.upper(), linewidth=2)
    ax.set_title(f'O3A Skill-Weighted Ensemble Annual Series: {variable.upper()}', fontweight='bold'); ax.set_xlabel('Year', fontweight='bold'); ax.set_ylabel('Precipitation' if variable == 'pr' else 'Temperature', fontweight='bold'); ax.grid(True, linestyle=':', alpha=0.5); ax.legend()
    fig.savefig(FIG_DIR / f'o3a_annual_series_{variable}.png', dpi=350, bbox_inches='tight'); fig.savefig(FIG_DIR / f'o3a_annual_series_{variable}.pdf', bbox_inches='tight'); plt.show()
print('[OK] figures saved to', FIG_DIR)

In [ ]:
# Cell 11 - Completion checklist and summary markdown
checks = []
required = [
    TABLE_DIR / 'o3a_model_pools_and_weights.csv',
    TABLE_DIR / 'o3a_zone_bias_correction_factors.csv',
    TABLE_DIR / 'o3a_zone_annual_ensemble_timeseries.csv',
    TABLE_DIR / 'o3a_period_changes_by_zone.csv',
    TABLE_DIR / 'o3a_spatial_change_inventory.csv',
    FIG_DIR / 'o3a_model_weights_by_variable.png',
]
for p in required:
    checks.append({'path': str(p), 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0})

for scenario in [HISTORICAL] + SCENARIOS:
    for variable in VARIABLES:
        p = NC_DIR / scenario / f'o3a_{variable}_{scenario}_skill_weighted_ensemble_annual.nc'
        checks.append({'path': str(p), 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0})

check_df = pd.DataFrame(checks)
check_df.to_csv(TABLE_DIR / 'o3a_completion_checklist.csv', index=False)

pool_txt = pd.read_csv(TABLE_DIR / 'o3a_model_pools_and_weights.csv').to_markdown(index=False)
summary_lines = [
    '# O3A Skill-Weighted Annual Projection Ensemble Summary',
    '',
    '## Purpose',
    '',
    'This workflow implements the first stage of Objective 3 by constructing annual, zone-bias-corrected, skill-weighted CMIP6 projection ensembles for precipitation, Tmax, and Tmin under SSP2-4.5 and SSP5-8.5.',
    '',
    '## Method Scope',
    '',
    '- Uses annual CMIP6 projection files from `output/cmip6/`.',
    '- Uses O2-passing models with available annual files for each variable.',
    '- Applies zone-specific annual bias correction using ERA5 and CMIP6 historical 1985-2014 zone means.',
    '- Applies O2 variable-specific skill-score weights.',
    '- Saves ensemble mean and 5th-95th percentile uncertainty bounds.',
    '',
    'This stage does not yet claim completion of the full PINN/RF/XGBoost/Ridge/BMA system proposed for the complete O3 objective.',
    '',
    '## Model Weights',
    '',
    pool_txt,
    '',
    '## Main Outputs',
    '',
    '- Model weights: `tables/o3a_model_pools_and_weights.csv`',
    '- Bias-correction factors: `tables/o3a_zone_bias_correction_factors.csv`',
    '- Zone annual ensemble time series: `tables/o3a_zone_annual_ensemble_timeseries.csv`',
    '- Period changes by zone: `tables/o3a_period_changes_by_zone.csv`',
    '- Spatial change inventory: `tables/o3a_spatial_change_inventory.csv`',
    '- Ensemble NetCDFs: `netcdf/`',
    '- Optional GeoTIFFs: `geotiff/`',
    '- Figures: `figures/`',
    '- Logs: `logs/`',
    '',
    '## Recommended Next Step',
    '',
    'After checking these outputs, write O3A Methods and Results. Then decide whether to proceed to O3B daily QDM repair or O3C stacking/PINN experiments.',
    '',
]
SUMMARY_PATH.write_text('\n'.join(summary_lines), encoding='utf-8')
print(check_df)
print(f'[OK] saved {TABLE_DIR / "o3a_completion_checklist.csv"}')
print(f'[OK] saved {SUMMARY_PATH}')